# Chapter 36: Linear and Logistic Regression for Prediction

Synthetic NRG shipment data illustrate numeric and probability prediction.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.predictive_regression import *
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(36)
n=240
distance=rng.uniform(1,20,n)
congestion=rng.integers(0,2,n)
hours=16+0.8*distance+5*congestion+rng.normal(0,2,n)
late=(hours>31).astype(int)
cut=180
print(f'Train rows: {cut}; test rows: {n-cut}; test late rate: {late[cut:].mean():.1%}')


Train rows: 180; test rows: 60; test late rate: 16.7%


In [ ]:
x=np.column_stack([distance,congestion]).tolist()
means,scales=standardize_fit(x[:cut])
z_train=standardize_apply(x[:cut],means,scales)
z_test=standardize_apply(x[cut:],means,scales)
lin_b,lin_w=linear_fit(z_train,hours[:cut],steps=3000,l2=.01)
hat=linear_predict(z_test,lin_b,lin_w)
print(f'Test MAE: {mean_absolute_error(hours[cut:],hat):.2f} hours')


Test MAE: 1.50 hours


In [ ]:
log_b,log_w=logistic_fit(z_train,late[:cut].tolist(),steps=4000,l2=.03)
prob=probability_predict(z_test,log_b,log_w)
print(f'Test log loss: {log_loss(late[cut:].tolist(),prob):.3f}')


Test log loss: 0.253


In [ ]:
residual=hours[cut:]-np.array(hat)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(hat,residual,alpha=.7); axes[0].axhline(0,color='black',lw=1); axes[0].set(xlabel='Predicted hours',ylabel='Residual',title='Linear residuals')
bins=np.linspace(0,1,6); ids=np.digitize(prob,bins)-1
pred=[]; obs=[]
for j in range(5):
 mask=ids==j
 if mask.any(): pred.append(np.mean(np.array(prob)[mask])); obs.append(np.mean(late[cut:][mask]))
axes[1].plot([0,1],[0,1],'--',color='grey'); axes[1].plot(pred,obs,'o-'); axes[1].set(xlabel='Mean probability',ylabel='Observed rate',title='Calibration check')
fig.tight_layout(); plt.show()


## Interpretation

The linear model is evaluated in hours and inspected through residuals. The logistic model is evaluated as a probability model before any action threshold is chosen. Both transformations were fitted using training data only.


In [ ]:
# Practice: change regularisation and compare unseen-data results.
